In [1]:
!pip install git+https://github.com/NREL/OCHRE.git

!pip install "numpy<2.0"

!pip uninstall gurobipy -y
!pip install gurobipy==12.0.0

print("\nRestart Session")

  Cloning https://github.com/NREL/OCHRE.git to /tmp/pip-req-build-cs3wffm2
  Running command git clone --filter=blob:none --quiet https://github.com/NREL/OCHRE.git /tmp/pip-req-build-cs3wffm2
  Resolved https://github.com/NREL/OCHRE.git to commit 0bf050355048a92301ece3122f8e5043889ad81c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Found existing installation: gurobipy 12.0.0
Uninstalling gurobipy-12.0.0:
  Successfully uninstalled gurobipy-12.0.0
  Using cached gurobipy-12.0.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (15 kB)
Using cached gurobipy-12.0.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (14.2 MB)

Restart Session


In [2]:
import os
import datetime as dt
from ochre import Dwelling
from ochre.utils import default_input_path
from pvlib.iotools import read_epw
import pandas as pd
import numpy as np

dwelling = Dwelling(
    start_time=dt.datetime(2018, 1, 1, 0, 0),
    time_res=dt.timedelta(minutes=60),
    duration=dt.timedelta(days=365),
    hpxml_file=os.path.join(default_input_path, 'Input Files', 'bldg0112631-up11.xml'),
    weather_file=os.path.join(default_input_path, 'Weather', 'USA_CO_Denver.Intl.AP.725650_TMY3.epw'),
)

df, metrics, hourly = dwelling.simulate()


# EXPORTS

# 1. Ηλεκτρισμός
pd.DataFrame({
    'datetime': df.index,
    'load_W': np.minimum(df['Total Electric Power (kW)'].values * 1000, 8000)
}).to_csv('ELECTRICITY_DENVER.csv', index=False)

# 2. Ζεστό Νερό
pd.DataFrame({
    'datetime': df.index,
    'dhw_W': np.minimum(df['Water Heating Electric Power (kW)'].values * 1000, 8000)
}).to_csv('DHW_DENVER.csv', index=False)

# 3. Θέρμανση Χώρου
pd.DataFrame({
    'datetime': df.index,
    'Q_space_W': np.minimum(df['HVAC Heating Electric Power (kW)'].values * 1000, 8000)
}).to_csv('SPACE_HEAT_DENVER.csv', index=False)

# Ανάγνωση καιρού
weather_file = os.path.join(default_input_path, "Weather", "USA_CO_Denver.Intl.AP.725650_TMY3.epw")
weather_df, _ = read_epw(weather_file)
weather_df = weather_df.iloc[:len(df)]

# 4. Solar Data (GHI)
pd.DataFrame({
    'datetime': df.index,
    'GHI': weather_df['ghi'].values
}).to_csv('SOLAR_DATA_DENVER.csv', index=False)

# 5. Temperatures (Ambient & Collector)
pd.DataFrame({
    'datetime': df.index,
    'Tamb_C': weather_df['temp_air'].values + 7,
    'Tcoll_C': weather_df['temp_air'].values + 20
}).to_csv('TEMPERATURES_DENVER.csv', index=False)

print("OK")

2026-03-27 12:16:09.011524 - ochre at 2018-01-01 00:00:00: Initializing ochre (OCHRE v0.9.2)
2026-03-27 12:16:16.327396 - ochre at 2018-01-01 00:00:00: Dwelling Initialized
2026-03-27 12:16:16.328193 - ochre at 2018-01-01 00:00:00: Running Simulation for 365 days, 0:00:00
2026-03-27 12:16:47.890148 - ochre at 2019-01-01 00:00:00: Simulation complete, time series results saved to: /usr/local/lib/python3.12/dist-packages/ochre/defaults/Input Files/ochre.csv
2026-03-27 12:16:47.895425 - ochre at 2019-01-01 00:00:00: Post-processing metrics saved to: /usr/local/lib/python3.12/dist-packages/ochre/defaults/Input Files/ochre_metrics.csv
2026-03-27 12:16:48.068375 - ochre at 2019-01-01 00:00:00: Hourly results saved to: /usr/local/lib/python3.12/dist-packages/ochre/defaults/Input Files/ochre_hourly.csv
OK
